In [1]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.base import clone
from sklearn.model_selection import ShuffleSplit
from scipy.stats import mode


In [2]:
number_of_trees=1000
number_of_samples=100

In [3]:
X,y = make_moons(n_samples=1000,noise=0.4)

In [6]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
tree_pipeline= make_pipeline(StandardScaler(),PCA())
X_train_tree=tree_pipeline.fit_transform(X_train)
X_test_tree=tree_pipeline.transform(X_test)

In [ ]:
params = {
    'max_leaf_nodes': list(range(2, 100)),
    'max_depth': [1, 2, 3, 4, 5, 6],
    'min_samples_split': [2, 3, 4]
}
gridsearchcv = GridSearchCV(estimator=DecisionTreeClassifier(random_state=42),param_grid=params,cv=3)

gridsearchcv.fit(X_train_tree,y_train)

In [7]:
best_tree=DecisionTreeClassifier(max_depth=6, max_leaf_nodes=17, random_state=42)
best_tree.fit(X_train_tree,y_train)
print(accuracy_score(y_test,best_tree.predict(X_test_tree)))


0.86


In [8]:
X_train_sub=[]
y_train_sub=[]
mini_sets=[]
ss=ShuffleSplit(n_splits=number_of_trees,train_size=number_of_samples,random_state=42)
for subset_idx,_ in ss.split(X_train):
    X_train_sub.append(X_train[subset_idx])
    y_train_sub.append(y_train[subset_idx])
    mini_sets.append((X_train_sub,y_train_sub))
    

In [9]:
forest=[clone(best_tree) for _ in range (number_of_trees)]
accuracy_scores=[]
y_preds=[]
for tree,X,y in zip(forest,X_train_sub,y_train_sub):
    tree.fit(X,y)
    y_preds.append(tree.predict(X_test))
    accuracy_scores.append(accuracy_score(y_test,tree.predict(X_test)))



In [10]:
result=mode(y_preds,axis=0)
final_vote=result.mode.reshape([-1])
print(accuracy_score(y_test,final_vote))

0.87
